# Geometric Brownian Motion & Ito Calculus

From-scratch exploration of Brownian motion, Ito's lemma, and GBM -- the workhorse model of quantitative finance.

**Why this matters:** Every option pricing formula, every Monte Carlo risk engine, and every quantitative trading model rests on the ideas in this notebook. GBM is the "hello world" of mathematical finance -- simple enough to solve exactly, rich enough to be genuinely useful.

> **CFA Exam Tip:** The CFA curriculum assumes GBM when discussing lognormal asset returns, the Black-Scholes-Merton model, and Monte Carlo simulation. Understanding the mechanics here directly supports the Derivatives and Portfolio Management readings.

**Prerequisites:** Basic probability (normal distribution, expectation, variance), calculus (derivatives, integrals, chain rule). No prior knowledge of stochastic calculus is assumed -- we build it from scratch.

**Outline**
1. How do stock prices move?
2. Random walks and coin flips
3. Brownian motion -- the continuous limit
4. Ito's lemma -- calculus with randomness
5. Geometric Brownian Motion
6. Solving the GBM SDE
7. Euler-Maruyama simulation
8. Exact simulation
9. Why log-returns are normal
10. Path properties and fan charts
11. Calibration to data
12. Summary and key takeaways
13. References

---
## 1. How Do Stock Prices Move?

If you watch a stock ticker for a few minutes, you notice something peculiar: the price jiggles up and down in a way that looks completely unpredictable. One moment it ticks up by a cent, the next it drops by two cents. Over a day, these tiny movements accumulate into a net gain or loss.

This raises a fundamental question: **is there a mathematical model that captures this kind of random, jittery movement?**

The answer is **Geometric Brownian Motion (GBM)** -- a model introduced to finance by Paul Samuelson in 1965 and made famous by Black, Scholes, and Merton in 1973. It remains the default model for stock prices in quantitative finance today.

### The random walk story

Picture yourself standing on a number line at position zero. Every second, you flip a coin:
- **Heads:** step right (+1)
- **Tails:** step left (-1)

After 100 flips, where are you? You could be anywhere from -100 to +100, but most likely you are somewhere near zero, within about 10 steps in either direction ($\sqrt{100} = 10$). This is a **random walk** -- the simplest model of randomness evolving over time.

Now imagine doing this not once per second, but a thousand times per second, with tiny steps. The path becomes smoother, more continuous, but still inherently random. In the limit, you get **Brownian motion** -- a continuous version of the coin-flip process.

Finally, instead of adding random steps to a position, multiply the position by random factors (so the randomness is *proportional* to where you are). That gives you GBM -- and it is how stock prices behave.

But to understand GBM, we need to build up from simpler ideas. Our journey:

1. **Coin flips** -- the simplest random process
2. **Random walks** -- coin flips accumulated over time
3. **Brownian motion** -- what happens when you take infinitely many, infinitely small steps
4. **Ito's lemma** -- the special calculus rules needed for random processes
5. **GBM** -- applying all of the above to model stock prices

Think of it like building a house: coin flips are the bricks, Brownian motion is the foundation, Ito's lemma is the engineering, and GBM is the finished structure.

> **Key Concept:** Brownian motion is the "continuous coin flip." Just as the Central Limit Theorem says that many small random effects sum to a normal distribution, Donsker's theorem says that many small random steps converge to Brownian motion. It is the universal model for accumulated small random shocks.### A Brief History of Random Walks in Finance

| Year | Contribution | Key idea |
|:-----|:------------|:---------|
| 1900 | Bachelier, *Théorie de la Spéculation* | First model of stock prices as random walks |
| 1923 | Wiener | Rigorous construction of continuous Brownian motion |
| 1944 | Itô | Stochastic calculus — how to do calculus with random functions |
| 1965 | Samuelson | GBM for stock prices (fixing Bachelier's negative price problem) |
| 1973 | Black, Scholes, Merton | Option pricing formula derived from GBM |

> **Key Concept:** The "random walk hypothesis" states that future price changes are unpredictable from past changes. If this is true (approximately), then Brownian motion is the natural continuous-time model. This doesn't mean prices are "random" in a nihilistic sense — it means markets are **informationally efficient**: all known information is already reflected in the price.


## 2. Setup

We import NumPy for numerical computation, SciPy for statistical distributions, and Matplotlib for visualization. All algorithms are implemented from scratch -- no black-box libraries.

We also set a random seed for reproducibility. Every time you run this notebook with the same seed, you will get identical random paths -- useful for debugging and comparison.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 3. Random Walks and Coin Flips

### The simplest random process

Imagine flipping a fair coin repeatedly. Each flip, you either go up one step (+1) or down one step (-1), each with probability 1/2. After $n$ flips, your position is:

$$S_n = X_1 + X_2 + \cdots + X_n$$

where each $X_i = +1$ or $-1$ with equal probability.

This is a **symmetric random walk**. Let us think about its properties:

- **Average position:** $E[S_n] = 0$ (the walk has no drift -- up and down are equally likely)
- **Spread:** $\text{Var}(S_n) = n$, so $\text{Std}(S_n) = \sqrt{n}$

> **Key Concept:** The standard deviation grows as $\sqrt{n}$, not $n$. This square-root scaling is one of the most important facts in probability. It means that after 100 flips, the typical deviation from zero is 10 (not 100). Randomness accumulates more slowly than you might expect.

### Why square-root scaling matters for finance

This is why we annualize volatility by multiplying daily volatility by $\sqrt{252}$ (not 252). If a stock's daily standard deviation is 1%, its annual standard deviation is roughly $1\% \times \sqrt{252} \approx 15.9\%$, not $252\%$. The square root rule is a direct consequence of independent random increments.

> **CFA Exam Tip:** The $\sqrt{T}$ rule for scaling volatility appears frequently in exam questions. Daily VaR $\times \sqrt{h}$ gives $h$-day VaR, but only under the assumption of i.i.d. returns. Know this assumption and when it breaks down (e.g., volatility clustering).

### A worked example

Suppose we flip a coin 4 times and get: Heads (+1), Tails (-1), Tails (-1), Heads (+1).

| Flip | Result | Position |
|------|--------|----------|
| 1    | H (+1) | +1       |
| 2    | T (-1) |  0       |
| 3    | T (-1) | -1       |
| 4    | H (+1) |  0       |

After 4 flips we are back at 0. The theory says: $E[S_4] = 0$ (check!) and $\text{Std}(S_4) = \sqrt{4} = 2$, so being at 0 is well within the typical range.

### From random walk to Brownian motion -- the key idea

Here is the beautiful connection discovered by Monroe Donsker in 1951: if you take a random walk with $n$ steps, rescale time to fit in $[0, 1]$, and rescale the step size by $1/\sqrt{n}$, then as $n \to \infty$, the random walk converges to a smooth (but infinitely wiggly) curve called **Brownian motion**.

The next plot shows this convergence in action. Watch how the jagged walk becomes smoother-looking (yet still rough at every scale) as we increase the number of steps.### From Discrete to Continuous

The key insight is a **scaling limit**. If we:
1. Take more and more steps ($n \to \infty$)
2. Make each step smaller (proportional to $1/\sqrt{n}$)
3. Speed up time (each step takes $T/n$ seconds)

...the resulting path converges to **Brownian motion**. This is the **Donsker's theorem** (functional central limit theorem) — the continuous analogue of the CLT.

> **Key Concept:** Brownian motion is the continuous-time limit of a scaled random walk, just as the normal distribution is the limit of scaled sums (CLT). This connection gives Brownian motion its mathematical foundation.

Let's watch this convergence visually:


In [ ]:
# Visualize how a random walk converges to Brownian motion as we take more, smaller steps.
# As n increases, the jagged random walk looks increasingly like a smooth (but rough!) curve.

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, n in enumerate([10, 100, 1000, 10000]):
    ax = axes[idx]
    steps = rng.choice([-1, 1], size=n)
    walk = np.cumsum(steps) / np.sqrt(n)          # scale by 1/sqrt(n)
    t_walk = np.arange(n + 1) / n                 # rescale time to [0, 1]
    walk = np.concatenate([[0], walk])
    
    ax.plot(t_walk, walk, color=PRIMARY, linewidth=0.5 if n > 100 else 1.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'n = {n} steps')
    ax.set_xlabel('t')
    ax.set_ylabel('W(t)')

plt.suptitle('Random Walk to Brownian Motion (Donsker\'s Theorem)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:** With 10 steps, the walk is clearly jagged and discrete. By 10,000 steps, it looks like a continuous, wiggly curve. This is Brownian motion emerging from the random walk. The key insight: no matter how much we "zoom in" on a Brownian motion path, it always looks equally wiggly -- it is a fractal.

Notice that the vertical scale stays roughly the same across all four panels -- this is because of the $1/\sqrt{n}$ rescaling. Without it, the 10,000-step walk would be 100 times taller than the 10-step walk.

> **Key Concept:** This convergence is not just visual -- it is a rigorous mathematical theorem (Donsker's Invariance Principle). It says that *any* sequence of i.i.d. random variables with mean 0 and finite variance, when scaled by $1/\sqrt{n}$, converges to Brownian motion. The details of each step (coin flip, dice roll, etc.) do not matter -- only the mean and variance. This universality is why Brownian motion appears everywhere in nature and finance.> **Key Concept:** This convergence is not just visual — it's mathematically rigorous. Donsker's theorem guarantees that the scaled random walk converges *in distribution* to Brownian motion as the number of steps goes to infinity. The rate of convergence is $O(1/\sqrt{n})$ — the same rate as the CLT, which is no coincidence.

This limit is the bridge between the discrete binomial option pricing model and the continuous Black-Scholes model. As the binomial tree gets finer, it converges to BSM — precisely because the underlying random walk converges to Brownian motion.


---
## 4. Brownian Motion / Wiener Process

Now we define Brownian motion precisely. A **standard Brownian motion** (also called a Wiener process) $W_t$ is a random process that satisfies four properties:

1. **Starts at zero:** $W_0 = 0$
2. **Independent increments:** The change $W_t - W_s$ is independent of everything that happened before time $s$. Knowing the past does not help predict the future *change*.
3. **Normal increments:** $W_t - W_s \sim \mathcal{N}(0, t-s)$ for $t > s$. The change over any time interval is normally distributed, with variance equal to the length of the interval.
4. **Continuous paths:** The function $t \mapsto W_t$ is continuous (no jumps).

> **Key Concept:** Property 3 is the most important for computation. It tells us that $W_t \sim \mathcal{N}(0, t)$ -- at time $t$, the Brownian motion is a normal random variable with mean 0 and variance $t$. The uncertainty (standard deviation) grows as $\sqrt{t}$.

### Real-world analogy: pollen in water

Brownian motion was first observed by botanist Robert Brown in 1827, who watched pollen grains jiggling under a microscope. Each grain was being bombarded by millions of invisible water molecules from random directions. The net effect of all these tiny, random pushes is a smooth-looking but erratic path -- exactly what our random walk converges to.

### A numerical check: the properties in practice

Suppose we simulate a Brownian motion path at $t = 0, 0.01, 0.02, \ldots, 1.0$. Then:
- Each increment $W_{t+0.01} - W_t$ should be $\sim \mathcal{N}(0, 0.01)$, i.e., have std $= \sqrt{0.01} = 0.1$.
- The increment from $t=0$ to $t=0.5$ should be $\sim \mathcal{N}(0, 0.5)$, i.e., have std $= \sqrt{0.5} \approx 0.707$.
- Any two non-overlapping increments (e.g., $W_{0.3} - W_{0.1}$ and $W_{0.7} - W_{0.5}$) should be independent.

### Remarkable properties of Brownian motion

- **Nowhere differentiable:** Despite being continuous, Brownian motion has no derivative at any point. The path is infinitely "jagged" at every scale.
- **Quadratic variation = $t$:** Over $[0, T]$, $\sum (\Delta W)^2 \to T$. This is crucial for Ito's lemma.
- **Fractal dimension 1.5:** A Brownian path in 2D has fractal dimension 1.5 (between a line and a plane).

> **Key Concept:** The quadratic variation property -- $(dW)^2 = dt$ -- is the single most important fact for stochastic calculus. In ordinary calculus, $(dx)^2 \approx 0$ and can be ignored. In stochastic calculus, $(dW)^2 = dt \neq 0$, and ignoring it gives the wrong answer. This is exactly why Ito's lemma has an extra term.

### How to simulate Brownian motion

From Property 3: $W_{t+\Delta t} - W_t \sim \mathcal{N}(0, \Delta t)$. So we can simulate by:
1. Choose a small time step $\Delta t = T/n$.
2. Generate independent $\Delta W_i \sim \mathcal{N}(0, \Delta t) = \sqrt{\Delta t} \cdot Z$ where $Z \sim \mathcal{N}(0,1)$.
3. Accumulate: $W_{i+1} = W_i + \Delta W_i$.

The following code implements this and verifies that $W_t \sim \mathcal{N}(0, t)$ by simulating 10,000 paths.### Why Brownian Motion Is "Nowhere Differentiable"

A remarkable mathematical property: although Brownian paths are continuous, they are **nowhere differentiable** — at no point does the path have a well-defined slope. Intuitively, the path is so jagged at every scale that you can never draw a tangent line.

This has a profound consequence for finance: **you cannot predict the next instant's price move from the current trajectory.** No matter how closely you zoom in, the path looks equally jagged. This is the mathematical embodiment of market unpredictability.

> **Key Concept:** The quadratic variation of Brownian motion over $[0, T]$ is $T$ (not zero, as it would be for a smooth function). This finite quadratic variation is what makes Itô calculus different from ordinary calculus — and is the root cause of the $-\sigma^2/2$ correction in GBM.


In [ ]:
def simulate_brownian_motion(T, n_steps, n_paths=1):
    """Simulate standard Brownian motion paths.
    
    The key idea: Brownian motion increments dW are independent N(0, dt) random
    variables.  We generate them all at once and take the cumulative sum.
    """
    dt = T / n_steps
    # Each increment is N(0, dt) = sqrt(dt) * N(0,1)
    dW = rng.normal(0, np.sqrt(dt), (n_paths, n_steps))
    W = np.zeros((n_paths, n_steps + 1))
    W[:, 1:] = np.cumsum(dW, axis=1)  # W_0 = 0, then accumulate increments
    t = np.linspace(0, T, n_steps + 1)
    return t, W

# ── Simulate 5 sample paths over [0, 1]
T, n_steps, n_paths = 1.0, 1000, 5
t, W = simulate_brownian_motion(T, n_steps, n_paths)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample paths
colors_bm = [PRIMARY, SECONDARY, TERTIARY, ACCENT, 'mediumpurple']
for i in range(n_paths):
    axes[0].plot(t, W[i], color=colors_bm[i], linewidth=0.8, alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('W(t)')
axes[0].set_title(f'Brownian Motion ({n_paths} paths)')

# Right: verify that W(t) ~ N(0, t) by looking at 10,000 paths at three time points
_, W_many = simulate_brownian_motion(T, n_steps, 10000)
for ti, c in [(0.25, PRIMARY), (0.5, SECONDARY), (1.0, TERTIARY)]:
    idx = int(ti * n_steps)
    axes[1].hist(W_many[:, idx], bins=80, density=True, alpha=0.4, color=c, label=f't={ti}')
    x = np.linspace(-3, 3, 200)
    axes[1].plot(x, stats.norm.pdf(x, 0, np.sqrt(ti)), color=c, linewidth=2)

axes[1].set_xlabel('W(t)')
axes[1].set_ylabel('Density')
axes[1].set_title('Distribution of W(t) ~ N(0, t)')
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretation:** On the left, each Brownian path wanders randomly. On the right, histograms from 10,000 paths confirm that at each time $t$, the values follow a $\mathcal{N}(0, t)$ distribution. Notice how the distribution at $t = 1$ (green) is wider than at $t = 0.25$ (blue) -- the spread grows as $\sqrt{t}$.

The theoretical PDFs (solid lines) overlay the histograms almost perfectly. This is a strong empirical confirmation that our simulation correctly implements the properties of Brownian motion.

Also observe that the paths on the left cross zero multiple times. This is a general property: a Brownian motion returns to zero infinitely often, yet the time between returns grows longer and longer. The probability of being positive at time $T$ is exactly 50% -- the process has no directional bias.> **Key Concept:** The distribution widens over time because $W_t \sim \mathcal{N}(0, t)$ — the variance grows linearly with time. After 1 year, the standard deviation is 1; after 4 years, it's 2 (not 4). This $\sqrt{t}$ scaling is fundamental and appears everywhere in finance (the "square-root-of-time" rule for volatility annualisation).

> **Common Mistake:** Brownian motion is NOT a good model for stock prices directly, because it can go negative. Stock prices must be positive. This is why we need Geometric Brownian Motion — applying the exponential function to Brownian motion ensures positivity.


### From Brownian motion to stock prices: the missing link

Brownian motion gives us a continuous random process, but it has two problems as a stock price model:
1. It can go negative (stock prices cannot).
2. A $\$1$ move means the same thing whether the stock is at $\$10$ or $\$1000$.

To fix these, we will build GBM by applying an exponential function to Brownian motion. But applying functions to random processes requires a special version of the chain rule -- **Ito's lemma** -- because the usual calculus rules give the wrong answer when randomness is involved.

The next section develops Ito's lemma carefully, starting from the intuition of why ordinary calculus fails and building up to the full formula.To go from Brownian motion (which can be negative) to stock prices (which must be positive), we need a transformation. The key question is: should we model price *levels* or price *changes*?

| Model | Equation | Problem |
|:------|:---------|:--------|
| Arithmetic BM | $S_t = S_0 + \mu t + \sigma W_t$ | $S_t$ can be negative! |
| Geometric BM | $dS/S = \mu \, dt + \sigma \, dW$ | $S_t > 0$ always (exponential) |

GBM models **percentage changes** (returns), not dollar changes. This is the right model because a \$1 move on a \$10 stock (10%) is very different from a \$1 move on a \$1000 stock (0.1%).


---
## 5. Ito's Lemma -- When Calculus Meets Randomness

### The problem

In ordinary calculus, if you know how a variable $x$ changes ($dx$), and you have a function $f(x)$, then the chain rule tells you:

$$df = f'(x) \, dx$$

Simple. But what happens when $x$ is a random process driven by Brownian motion?

### Why the ordinary chain rule fails

Suppose $X_t$ follows a stochastic differential equation (SDE):

$$dX_t = \mu(X_t, t) \, dt + \sigma(X_t, t) \, dW_t$$

This says: $X_t$ changes by a predictable amount ($\mu \, dt$, the **drift**) plus a random amount ($\sigma \, dW_t$, the **diffusion**).

Now, if we apply a smooth function $f(X_t)$, the ordinary chain rule would give $df = f' \, dX$. But this is **wrong** for stochastic processes!

The reason: in ordinary calculus, $(dx)^2$ is negligible. But for Brownian motion, $(dW)^2 = dt$ -- it is **not** negligible! This creates a correction.

### Ito's Lemma (statement)

If $X_t$ satisfies $dX = \mu \, dt + \sigma \, dW$ and $f(x, t)$ is twice-differentiable, then:

$$df = \left(rac{\partial f}{\partial t} + \mu rac{\partial f}{\partial x} + rac{1}{2} \sigma^2 rac{\partial^2 f}{\partial x^2}ight) dt + \sigma rac{\partial f}{\partial x} \, dW$$

> **Key Concept:** The extra term $rac{1}{2}\sigma^2 f''$ is the **Ito correction**. It has no analogue in ordinary calculus. Intuitively: when you apply a curved function to a noisy process, the noise itself creates a systematic bias because up-moves and down-moves do not cancel out on a curve. This correction is why the expected return of a stock and the growth rate of the median stock price are different.

### Variable definitions

| Symbol | Meaning |
|--------|---------|
| $f(x, t)$ | Any smooth function applied to the process |
| $\mu$ | Drift coefficient (predictable part of change) |
| $\sigma$ | Diffusion coefficient (random part of change) |
| $\partial f / \partial x$ | First derivative of $f$ |
| $\partial^2 f / \partial x^2$ | Second derivative (curvature) -- creates the Ito correction |

### A worked example: $f(W_t) = W_t^2$

Apply Ito's lemma to $f(x) = x^2$ applied to $X_t = W_t$ (so $\mu = 0$, $\sigma = 1$).

- $f'(x) = 2x$, $f''(x) = 2$
- Ito gives: $d(W_t^2) = (0 + 0 + rac{1}{2} \cdot 1 \cdot 2) dt + 1 \cdot 2W_t \, dW_t = dt + 2W_t \, dW_t$

Integrating: $W_t^2 = t + 2\int_0^t W_s \, dW_s$, so $\int_0^t W_s \, dW_s = (W_t^2 - t)/2$.

Notice the surprising $-t/2$ term. In ordinary calculus, $\int x \, dx = x^2/2$. But stochastically, $\int W \, dW = (W^2 - t)/2$. The extra $-t/2$ is the Ito correction.

Let us verify this numerically.

### Ito's lemma: a gentle intuition

Why does the extra $\frac{1}{2}\sigma^2 f''$ term appear? Think of it this way:

Imagine a curved function $f(x) = x^2$. If $x$ is at zero and gets a random shock of $+1$ or $-1$ (equally likely), the average of $f$ is:

$$\frac{f(+1) + f(-1)}{2} = \frac{1 + 1}{2} = 1$$

But $f(0) = 0$! So even though the shocks average to zero, the function value *increased* on average. This is because $f$ curves upward -- both positive and negative shocks push $f$ up. The curvature term $f'' > 0$ captures this systematic bias.

Now consider $f(x) = e^x$ (the exponential, which we will use for GBM). At $x = 0$:
- $f(+0.1) = e^{0.1} \approx 1.1052$
- $f(-0.1) = e^{-0.1} \approx 0.9048$
- Average: $\approx 1.0050$, but $f(0) = 1.0000$.

The average is 0.50% higher than the "no-shock" value. This 0.50% is exactly $\frac{1}{2}(0.1)^2 f''(0) = \frac{1}{2}(0.01)(1) = 0.005$. This is the Ito correction in action.

For GBM, this curvature effect is exactly the **volatility drag** -- the reason the median stock price grows slower than the expected price. Higher volatility means more curvature effect, more drag.

> **CFA Exam Tip:** The relationship between arithmetic return ($\mu$) and geometric return ($\mu - \sigma^2/2$) is tested on the CFA exam. The geometric return is always less than the arithmetic return by approximately $\sigma^2/2$. This is a direct consequence of Ito's lemma applied to GBM.> **Key Concept:** In ordinary calculus, $d(f(x)) = f'(x) \, dx$ and we ignore higher-order terms like $(dx)^2$ because they vanish. But for stochastic processes, $(dW)^2 = dt$ (not zero!). This non-zero "quadratic variation" is what creates the extra $\frac{1}{2}\sigma^2 f''$ term. It's not an approximation — it's exact.

### The Most Important Application: $f(W) = W^2$

Apply Itô's lemma to $f(x) = x^2$ with $X_t = W_t$ ($\mu = 0$, $\sigma = 1$):

$$d(W_t^2) = 2W_t \, dW_t + \frac{1}{2}(2)(1)^2 \, dt = 2W_t \, dW_t + dt$$

Integrating: $W_T^2 = 2\int_0^T W_t \, dW_t + T$, or equivalently:

$$\int_0^T W_t \, dW_t = \frac{W_T^2 - T}{2}$$

In ordinary calculus, $\int x \, dx = x^2/2$. The Itô integral adds the correction $-T/2$. Let's verify this numerically:


In [ ]:
# ── Numerical verification of Ito's formula: int_0^t W_s dW_s = (W_t^2 - t)/2
n_steps_ito = 10000
dt = T / n_steps_ito
dW = rng.normal(0, np.sqrt(dt), n_steps_ito)
W_path = np.concatenate([[0], np.cumsum(dW)])
t_ito = np.linspace(0, T, n_steps_ito + 1)

# Numerical Ito integral: sum of W_{t_i} * Delta W_i  (left-point rule = Ito convention)
ito_integral = np.cumsum(W_path[:-1] * dW)
ito_integral = np.concatenate([[0], ito_integral])

# Analytical result from Ito's lemma
analytic = 0.5 * (W_path**2 - t_ito)

fig, ax = plt.subplots()
ax.plot(t_ito, ito_integral, color=PRIMARY, linewidth=1.5, label='Numerical Ito integral')
ax.plot(t_ito, analytic, '--', color=SECONDARY, linewidth=1.5, label='$(W_t^2 - t)/2$ (analytic)')
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.set_title("Verifying Ito's Formula: $\\int_0^t W_s \\, dW_s = (W_t^2 - t)/2$")
ax.legend()
plt.tight_layout()
plt.show()

max_error = np.max(np.abs(ito_integral - analytic))
print(f"Max approximation error: {max_error:.6f}")

**Interpretation:** The numerical Ito integral (blue) closely matches the analytical formula (orange dashed). The small discrepancy is due to the finite step size. As we make $\Delta t$ smaller, the two curves converge perfectly.

This verification is important: it confirms that the Ito integral $\int W \, dW$ truly equals $(W^2 - t)/2$, not $W^2/2$ as ordinary calculus would suggest. The missing $-t/2$ is a concrete, measurable consequence of the Ito correction.

**A deeper point:** The Ito integral uses the *left-endpoint* rule ($W_{t_i} \cdot \Delta W_i$), which gives a different answer than the midpoint rule (Stratonovich integral). The choice of convention matters! Finance universally uses the Ito convention because it produces non-anticipating integrals -- the integrand does not "peek into the future."This verification is important: it confirms that the Itô integral behaves differently from a Riemann integral. The $-T/2$ correction is real and measurable. In finance, this same correction manifests as the $-\sigma^2/2$ term in the GBM solution — without it, simulated stock prices would systematically drift upward too fast.


---
## 6. Geometric Brownian Motion -- The Standard Model for Stock Prices

### Motivation: why not use plain Brownian motion for stock prices?

Brownian motion itself is a terrible model for stock prices because:
1. **It can go negative.** A stock at   could drop to - . Stocks cannot go below zero.
2. **The same dollar move means different things.** A \ move when the stock is at   (10%) should be more significant than at   (0.2%). Brownian motion treats both identically.

### The fix: make the randomness proportional to the price

$$rac{dS}{S} = \mu \, dt + \sigma \, dW \quad 	ext{equivalently:} \quad dS = \mu S \, dt + \sigma S \, dW$$

> **Key Concept:** The left-hand side $dS/S$ is the *percentage* change, not the dollar change. A stock at   might move \/bin/bash.10, while one at   might move \ -- but both represent the same 1% move. This **multiplicative noise** is natural for financial prices.

### What each parameter means

| Parameter | Symbol | Meaning | Typical value |
|-----------|--------|---------|---------------|
| Drift | $\mu$ | Expected return per year (the trend) | 5-15% |
| Volatility | $\sigma$ | Std dev of returns per year (the randomness) | 15-40% |

### Real-world analogy

Think of $\mu$ as the speed of a river current and $\sigma$ as the turbulence. Both scale with where you are -- a wider river (higher price) means bigger waves.

### Solving the GBM SDE with Ito's lemma

**Trick:** Apply $f(S) = \ln S$. Using Ito's lemma with $f' = 1/S$ and $f'' = -1/S^2$:

$$d(\ln S) = \left(\mu - rac{\sigma^2}{2}ight) dt + \sigma \, dW$$

Integrating: $oxed{S_T = S_0 \exp\left[\left(\mu - rac{\sigma^2}{2}ight) T + \sigma W_Tight]}$

> **Key Concept:** The $-\sigma^2/2$ term is the **Ito correction** (volatility drag). The *median* stock price grows slower than the *expected* price. With $\mu = 10\%$, $\sigma = 20\%$: expected growth is 10%/yr, but median is only $10\% - 2\% = 8\%$/yr. The missing 2% is "lost" to randomness.

### Worked example with numbers

Let $S_0 = 100$, $\mu = 0.10$, $\sigma = 0.20$, $T = 1$ year.

- **Log drift:** $\mu - \sigma^2/2 = 0.10 - 0.02 = 0.08$
- **Expected price:** $E[S_T] = 100 	imes e^{0.10} = 110.52$
- **Median price:** $100 	imes e^{0.08} = 108.33$
- Expected > Median because the lognormal distribution has a long right tail.### The GBM Solution: Applying Itô to $\ln S$

To solve $dS/S = \mu \, dt + \sigma \, dW$, apply Itô's lemma to $f(S) = \ln S$:

$$d(\ln S) = \frac{1}{S} dS - \frac{1}{2} \frac{1}{S^2}(\sigma S)^2 dt = \left(\mu - \frac{\sigma^2}{2}\right) dt + \sigma \, dW$$

Integrating from $0$ to $T$:

$$\ln S_T - \ln S_0 = \left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T$$

$$\boxed{S_T = S_0 \exp\left[\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T\right]}$$

> **Key Concept:** The $-\sigma^2/2$ term is the **Itô correction** (also called "volatility drag" or "convexity adjustment"). It means the expected *log* return is $\mu - \sigma^2/2$, which is LESS than $\mu$. Higher volatility reduces compound growth — this is the mathematical basis of the "volatility drag" concept from portfolio theory.

> **Common Mistake:** Many students write $S_T = S_0 e^{\mu T + \sigma W_T}$ (missing the $-\sigma^2/2$). This is wrong and leads to $E[S_T] = S_0 e^{(\mu + \sigma^2/2)T}$ instead of the correct $E[S_T] = S_0 e^{\mu T}$.
> **CFA Exam Tip:** The CFA curriculum distinguishes between:
> - **Continuously compounded return** (log return): $r_c = \ln(S_T/S_0)$ — normally distributed under GBM
> - **Simple return** (discrete): $R = S_T/S_0 - 1$ — lognormally distributed under GBM
>
> Log returns are additive across time ($r_{0,T} = r_{0,1} + r_{1,2} + \cdots$), while simple returns are multiplicative ($1 + R_{0,T} = (1+R_{0,1})(1+R_{1,2})\cdots$). This additivity makes log returns convenient for statistical modelling.


### Why percentage changes are the right model

Consider two stocks:
- **Stock A** at $\$10$: a $\$1$ move is a 10% change -- huge!
- **Stock B** at $\$1000$: a $\$1$ move is a 0.1% change -- negligible.

If both stocks have the same fundamental riskiness, Stock B should have dollar moves 100 times larger than Stock A. This is exactly what GBM does: the noise term $\sigma S \, dW$ scales with the price $S$.

In percentage terms: both stocks have the same $dS/S = \mu \, dt + \sigma \, dW$. This makes GBM a **multiplicative** model -- returns compound, just like interest.

> **Key Concept:** GBM produces lognormally distributed prices, which means: (1) prices are always positive, (2) percentage returns are normally distributed, and (3) returns compound multiplicatively. These three properties match the basic stylized facts of stock prices.> **Key Concept:** This scale-invariance is why financial models use *returns* (percentage changes) rather than *price changes* (dollar amounts). A model for returns can be applied to any stock regardless of its price level. GBM captures this by modelling $dS/S$ (the return) rather than $dS$ (the price change).

> **CFA Exam Tip:** The CFA curriculum refers to the continuously compounded return as $\ln(S_T/S_0)$. Under GBM, this is normally distributed with mean $(\mu - \sigma^2/2)T$ and variance $\sigma^2 T$. The simple return $(S_T/S_0 - 1)$ is lognormally distributed.


In [ ]:
# GBM parameters
S0 = 100     # initial price
mu = 0.10    # drift (10% expected annual return)
sigma = 0.20 # volatility (20% annualized)
T_gbm = 1.0  # 1 year

print("GBM Parameters:")
print(f"  S_0 = {S0}, mu = {mu}, sigma = {sigma}, T = {T_gbm}")
print(f"  Log drift: mu - sigma^2/2 = {mu - sigma**2/2:.4f}")
print(f"  Expected price E[S_T] = {S0 * np.exp(mu * T_gbm):.2f}")
print(f"  Median price          = {S0 * np.exp((mu - sigma**2/2) * T_gbm):.2f}")

Now that we have the analytical solution, let us implement two simulation approaches: the approximate Euler-Maruyama method and the exact method. Comparing them illustrates when approximation errors matter.

Simulation is essential in practice because most derivatives have payoffs that depend on the *path* of the stock (e.g., Asian options, barrier options), not just the terminal price. Even when closed-form solutions exist, Monte Carlo simulation provides a flexible and general framework.

---
## 7. Euler-Maruyama Simulation

### What is Euler-Maruyama?

Just as the Euler method approximates ODEs, the **Euler-Maruyama** method approximates SDEs:

$$S_{t+\Delta t} = S_t + \mu S_t \Delta t + \sigma S_t \sqrt{\Delta t} \, Z, \quad Z \sim \mathcal{N}(0,1)$$

**Reading this:** Start at $S_t$, add drift $\mu S_t \Delta t$, add noise $\sigma S_t \sqrt{\Delta t} Z$.

### Worked example

$S_0 = 100$, $\mu = 0.10$, $\sigma = 0.20$, $\Delta t = 1/252$, $Z = 0.5$:

$S_1 = 100 + 0.10 	imes 100 	imes (1/252) + 0.20 	imes 100 	imes \sqrt{1/252} 	imes 0.5 = 100 + 0.04 + 0.63 = 100.67$

The drift contributed \/bin/bash.04 (tiny!) while the noise contributed \/bin/bash.63. **On short time scales, randomness dominates drift.** It takes months for the drift to become noticeable.

> **Key Concept:** Euler-Maruyama is simple but flawed for GBM: with large $\sigma$ or $\Delta t$, it can produce negative stock prices. The exact simulation (next section) avoids this.> **Important:** The Euler-Maruyama method discretises the SDE as:
> $$S_{t+\Delta t} = S_t + \mu S_t \Delta t + \sigma S_t \sqrt{\Delta t} \, Z$$
> where $Z \sim \mathcal{N}(0,1)$. This is a first-order approximation. For GBM, we're lucky — we have an exact solution and can simulate directly. But for more complex SDEs (stochastic volatility, jump-diffusion), Euler-Maruyama or its higher-order cousin (Milstein) may be the only option.

> **Common Mistake:** With Euler-Maruyama, the simulated price can go negative if $\Delta t$ is too large (the $\sigma S \sqrt{\Delta t} Z$ term can overwhelm $S_t$). The exact simulation avoids this because the exponential is always positive.


In [ ]:
def gbm_euler_maruyama(S0, mu, sigma, T, n_steps, n_paths=1):
    """Euler-Maruyama simulation of GBM.
    
    Steps through the SDE: dS = mu*S*dt + sigma*S*dW
    by replacing differentials with finite differences.
    """
    dt = T / n_steps
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_steps):
        Z = rng.standard_normal(n_paths)
        # Euler-Maruyama step: S_{i+1} = S_i * (1 + mu*dt + sigma*sqrt(dt)*Z)
        S[:, i+1] = S[:, i] * (1 + mu * dt + sigma * np.sqrt(dt) * Z)
    
    t = np.linspace(0, T, n_steps + 1)
    return t, S

# Simulate 5 paths over 1 year with daily steps
n_steps_em = 252  # daily time steps (252 trading days)
n_paths_em = 5
t_em, S_em = gbm_euler_maruyama(S0, mu, sigma, T_gbm, n_steps_em, n_paths_em)

fig, ax = plt.subplots()
for i in range(n_paths_em):
    ax.plot(t_em, S_em[i], linewidth=1.2)
ax.axhline(S0, color='black', linestyle=':', alpha=0.3)
ax.set_xlabel('Time (years)')
ax.set_ylabel('Stock Price')
ax.set_title('GBM Paths -- Euler-Maruyama')
plt.tight_layout()
plt.show()

**Interpretation:** Each path represents one possible future for the stock price. Some paths end above $100$ (drift wins), others below (noise overwhelms drift). With $\mu = 10\%$ and $\sigma = 20\%$, the drift adds about $\$0.04$ per day while the noise adds about $\pm \$1.26$ per day -- randomness dominates in the short term.

> **Key Concept:** On short time scales (days, weeks), stock price movements are dominated by randomness, not drift. The drift signal only becomes detectable over months or years. This is why short-term price prediction is so difficult.

To see this quantitatively: the signal-to-noise ratio of daily returns is $\mu \Delta t / (\sigma \sqrt{\Delta t}) = \mu \sqrt{\Delta t} / \sigma$. For daily data: $0.10 \times 0.063 / 0.20 = 0.032$. The signal is only 3.2% of the noise! You would need about $(1/0.032)^2 \approx 1000$ days (4 years) of data before the drift becomes statistically detectable.

---
## 8. Exact Simulation

Since we solved GBM in closed form, we can simulate *exactly*:

$$S_{t+\Delta t} = S_t \exp\left[\left(\mu - rac{\sigma^2}{2}ight) \Delta t + \sigma \sqrt{\Delta t} \, Zight]$$

**Advantages:** (1) Always positive (exponential is always positive). (2) Exact for any step size. (3) Faster.

### Worked example

Same parameters, $Z = 0.5$:
$S_1 = 100 	imes \exp(0.000317 + 0.006299) = 100 	imes e^{0.006617} = 100.66$

Very close to Euler ( .67) because $\Delta t$ is small. For large steps, the difference is significant.

In [ ]:
def gbm_exact(S0, mu, sigma, T, n_steps, n_paths=1):
    """Exact (lognormal) simulation of GBM.
    
    Uses the closed-form solution: S_{t+dt} = S_t * exp((mu - sigma^2/2)*dt + sigma*sqrt(dt)*Z)
    This is exact regardless of step size and always produces positive prices.
    """
    dt = T / n_steps
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0
    
    for i in range(n_steps):
        Z = rng.standard_normal(n_paths)
        S[:, i+1] = S[:, i] * np.exp((mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
    
    t = np.linspace(0, T, n_steps + 1)
    return t, S

# ── Compare Euler vs Exact at coarse and fine time steps
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for n_steps_cmp, ax_idx in [(10, 0), (252, 1)]:
    rng_copy = np.random.default_rng(42)
    dt = T_gbm / n_steps_cmp
    Z_shared = rng_copy.standard_normal(n_steps_cmp)  # same random numbers for fair comparison
    
    S_euler = np.zeros(n_steps_cmp + 1)
    S_exact = np.zeros(n_steps_cmp + 1)
    S_euler[0] = S_exact[0] = S0
    
    for i in range(n_steps_cmp):
        S_euler[i+1] = S_euler[i] * (1 + mu * dt + sigma * np.sqrt(dt) * Z_shared[i])
        S_exact[i+1] = S_exact[i] * np.exp((mu - 0.5*sigma**2) * dt + sigma * np.sqrt(dt) * Z_shared[i])
    
    t_cmp = np.linspace(0, T_gbm, n_steps_cmp + 1)
    axes[ax_idx].plot(t_cmp, S_euler, 'o-', color=PRIMARY, markersize=3, label='Euler-Maruyama')
    axes[ax_idx].plot(t_cmp, S_exact, 's-', color=SECONDARY, markersize=3, label='Exact')
    axes[ax_idx].set_xlabel('Time')
    axes[ax_idx].set_ylabel('S(t)')
    axes[ax_idx].set_title(f'dt = {dt:.4f} ({n_steps_cmp} steps)')
    axes[ax_idx].legend()

plt.suptitle('Euler-Maruyama vs Exact Simulation', fontsize=14)
plt.tight_layout()
plt.show()

**Interpretation:** With only 10 steps (left), the Euler-Maruyama and exact methods differ noticeably. With 252 steps (right), they are nearly indistinguishable.

The practical takeaway: for GBM specifically, always use the exact method. It is no more complex to implement, never produces negative prices, and is mathematically exact regardless of step size. Euler-Maruyama is useful for SDEs that do not have closed-form solutions (e.g., stochastic volatility models like Heston).

> **CFA Exam Tip:** When the CFA curriculum discusses Monte Carlo simulation for option pricing or VaR, it implicitly uses the exact GBM formula. Each simulated path draws $Z \sim \mathcal{N}(0,1)$ and computes $S_T = S_0 e^{(\mu - \sigma^2/2)T + \sigma\sqrt{T}Z}$. This is the "exact simulation" method.

**Why does Euler-Maruyama fail with few steps?** The error comes from ignoring the Ito correction within each step. The exact method incorporates the $-\sigma^2/2$ correction analytically, while Euler-Maruyama only gets it right in the limit of many steps.> **Key Concept:** The exact simulation is always preferable when available because:
> 1. No discretisation error (the solution is exact at every time step)
> 2. Prices are always positive (exponential of a normal)
> 3. Larger time steps can be used without loss of accuracy
>
> In practice, exact simulation is possible for GBM and a few other SDEs (Ornstein-Uhlenbeck, CIR with special care). For general SDEs, you must use Euler-Maruyama or Milstein.


---
## 9. Why Log-Returns Are Normal (Statistical Properties)

### The lognormal distribution

From our GBM solution, $S_T = S_0 \exp(X)$ where $X \sim \mathcal{N}(m, s^2)$ with $m = (\mu - \sigma^2/2)T$, $s = \sigma\sqrt{T}$.

The log-return is normally distributed:

$$\ln\frac{S_T}{S_0} \sim \mathcal{N}\left(\left(\mu - \frac{\sigma^2}{2}\right)T, \; \sigma^2 T\right)$$

> **Key Concept:** Practitioners work with **log-returns** because they are symmetric, additive (daily log-returns sum to weekly), and normally distributed under GBM. Simple returns ($S_T/S_0 - 1$) are skewed and harder to work with.

### Why log-returns and not simple returns?

Consider a stock that goes from $\$100$ to $\$150$ (simple return = +50%) and then back to $\$100$ (simple return = -33.3%). The simple returns do not add to zero! But the log-returns do: $\ln(150/100) + \ln(100/150) = 0$. This **additivity** makes log-returns the natural choice for multi-period analysis.

Additionally, log-returns are symmetric: a move from $\$100$ to $\$110$ has the same log-return magnitude as a move from $\$110$ to $\$100$ (about $\pm 9.5\%$). Simple returns give $+10\%$ and $-9.1\%$ -- an asymmetry that complicates analysis.

### Moments of the lognormal distribution

| Property | Formula | Value ($S_0=100, \mu=0.10, \sigma=0.20, T=1$) |
|----------|---------|-------|
| Mean | $E[S_T] = S_0 e^{\mu T}$ | 110.52 |
| Variance | $\text{Var}(S_T) = S_0^2 e^{2\mu T}(e^{\sigma^2 T} - 1)$ | 500.68 |
| Median | $S_0 e^{(\mu - \sigma^2/2)T}$ | 108.33 |

**Mean > Median** -- the lognormal is positively skewed. Most paths end below the average.

> **CFA Exam Tip:** The CFA exam tests the distinction between arithmetic and geometric returns. The geometric mean return corresponds to the median outcome under GBM (what a typical investor experiences), while the arithmetic mean corresponds to the expected value (pulled up by rare large gains). For long-horizon projections, the geometric mean is more appropriate.

In [ ]:
# ── Large simulation to verify analytical properties
n_mc = 100_000
Z_mc = rng.standard_normal(n_mc)
S_T = S0 * np.exp((mu - 0.5 * sigma**2) * T_gbm + sigma * np.sqrt(T_gbm) * Z_mc)

# Analytical values
E_ST = S0 * np.exp(mu * T_gbm)
Var_ST = S0**2 * np.exp(2 * mu * T_gbm) * (np.exp(sigma**2 * T_gbm) - 1)
Med_ST = S0 * np.exp((mu - 0.5 * sigma**2) * T_gbm)

print(f"{'Property':<20} {'Analytical':>12} {'Simulated':>12}")
print('-' * 46)
print(f"{'E[S_T]':<20} {E_ST:>12.4f} {np.mean(S_T):>12.4f}")
print(f"{'Var(S_T)':<20} {Var_ST:>12.4f} {np.var(S_T):>12.4f}")
print(f"{'Std(S_T)':<20} {np.sqrt(Var_ST):>12.4f} {np.std(S_T):>12.4f}")
print(f"{'Median(S_T)':<20} {Med_ST:>12.4f} {np.median(S_T):>12.4f}")

**Interpretation:** Simulated values closely match analytical formulas. The small discrepancies are Monte Carlo noise, decreasing as $1/\sqrt{n}$.

With 100,000 simulations, the standard error of the mean estimate is $\text{Std}(S_T)/\sqrt{100000} \approx 22.4/316 \approx \$0.07$. This is why Monte Carlo with 100K paths gives mean estimates accurate to within pennies.

Now let us visualize both distributions.These analytical properties are exactly what make GBM tractable:
- $E[S_T]$ depends only on $\mu$ (the drift) — not on $\sigma$
- $\text{Var}[S_T]$ depends on both $\mu$ and $\sigma$
- The distribution of $S_T$ is fully characterised by two parameters ($\mu, \sigma$) — estimation reduces to finding these two numbers


In [ ]:
# ── Visualize the price and log-return distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Terminal price S_T (lognormal)
ax1.hist(S_T, bins=150, density=True, color=PRIMARY, edgecolor='white', alpha=0.7)
s_range = np.linspace(S_T.min(), np.percentile(S_T, 99), 300)
log_mu = np.log(S0) + (mu - 0.5 * sigma**2) * T_gbm
log_sigma = sigma * np.sqrt(T_gbm)
pdf_lognorm = stats.lognorm.pdf(s_range, s=log_sigma, scale=np.exp(log_mu))
ax1.plot(s_range, pdf_lognorm, color=SECONDARY, linewidth=2, label='Lognormal PDF')
ax1.axvline(E_ST, color=ACCENT, linestyle='--', label=f'E[S_T]={E_ST:.1f}')
ax1.axvline(Med_ST, color=TERTIARY, linestyle='--', label=f'Median={Med_ST:.1f}')
ax1.set_xlabel('S_T')
ax1.set_ylabel('Density')
ax1.set_title('Terminal Price Distribution (Lognormal)')
ax1.legend()

# Right: Log returns (normal)
log_returns = np.log(S_T / S0)
ax2.hist(log_returns, bins=100, density=True, color=PRIMARY, edgecolor='white', alpha=0.7)
x_norm = np.linspace(log_returns.min(), log_returns.max(), 200)
ax2.plot(x_norm, stats.norm.pdf(x_norm, log_mu - np.log(S0), log_sigma), color=SECONDARY, linewidth=2)
ax2.set_xlabel('log(S_T/S_0)')
ax2.set_ylabel('Density')
ax2.set_title('Log Return Distribution (Normal)')

plt.tight_layout()
plt.show()

**Interpretation:**
- **Left (price distribution):** Lognormal with long right tail. Mean (gold) is right of median (green), reflecting positive skew. A few paths reach $\$200+$, pulling the mean above the median.
- **Right (log-return distribution):** Beautifully symmetric and normal, as theory predicts. The mean log-return is $\mu - \sigma^2/2 = 8\%$, not $10\%$, because of the Ito correction.

This dual view -- lognormal prices, normal log-returns -- is the foundation of modern quantitative finance. Option pricing, VaR calculations, and portfolio optimization all rely on this connection.

---
## 10. Path Properties and Fan Charts

### What a single path cannot tell you

A single path is just one realization. To understand the *range of possibilities*, we simulate thousands of paths and look at the envelope of outcomes.

A **fan chart** shows percentile bands: the 5th-to-95th band contains 90% of all paths.

> **Key Concept:** The fan widens over time because uncertainty accumulates. The width grows as $\sqrt{T}$ -- after 2 years, the uncertainty is $\sqrt{2} \approx 1.41$ times wider than after 1 year (not twice as wide). This is the square-root-of-time rule.

### Reading a fan chart

Think of a fan chart as a weather forecast for stock prices:
- The **median line** (50th percentile) is the "most likely" path -- where you'd bet if you had to pick one number.
- The **25th-75th band** is like "partly cloudy" -- a reasonable range of outcomes.
- The **5th-95th band** is like "possible but unlikely" -- extreme scenarios that still happen 10% of the time.

### Analytical percentiles

The $p$-th percentile at time $T$: $S_T^{(p)} = S_0 \exp[m + s \cdot \Phi^{-1}(p)]$

For the 5th percentile: $S_1^{(5)} = 100 \times e^{0.08 - 0.20 \times 1.645} = 100 \times e^{-0.249} = 77.95$.

This means: there is a 5% chance the stock falls below $\$77.95$ in one year. This is closely related to **Value at Risk (VaR)** -- the maximum loss at a given confidence level.> **Key Concept:** A single price path tells you nothing about the model parameters. The same $\mu$ and $\sigma$ can produce wildly different paths depending on the random draws. To understand the model's behaviour, we need to look at the **distribution** across many paths — which is exactly what the fan chart below shows.

> **Important:** Fan charts are widely used in central banking (Bank of England inflation fan charts) and risk management. The width of the fan at any time $T$ is proportional to $\sigma\sqrt{T}$ — another manifestation of the square-root-of-time rule.


In [ ]:
# ── Fan chart: 5000 paths with percentile bands
n_fan = 5000
t_fan, S_fan = gbm_exact(S0, mu, sigma, 2.0, 504, n_fan)

percentiles = [5, 25, 50, 75, 95]
S_pcts = np.percentile(S_fan, percentiles, axis=0)

fig, ax = plt.subplots()
# Faint sample paths for texture
for i in range(20):
    ax.plot(t_fan, S_fan[i], color='grey', alpha=0.08, linewidth=0.5)

# Percentile bands
ax.fill_between(t_fan, S_pcts[0], S_pcts[4], alpha=0.15, color=PRIMARY, label='5th-95th percentile')
ax.fill_between(t_fan, S_pcts[1], S_pcts[3], alpha=0.25, color=PRIMARY, label='25th-75th percentile')
ax.plot(t_fan, S_pcts[2], color=SECONDARY, linewidth=2, label='Median')
ax.plot(t_fan, S0 * np.exp(mu * t_fan), '--', color=ACCENT, linewidth=2, label=f'E[S_t] = S_0 exp(mu*t)')

ax.set_xlabel('Time (years)')
ax.set_ylabel('Stock Price')
ax.set_title(f'GBM Fan Chart ({n_fan} paths, mu={mu}, sigma={sigma})')
ax.legend()
plt.tight_layout()
plt.show()

**Interpretation:**
- The **expected value** (gold dashed) grows exponentially at rate $\mu$.
- The **median** (orange) is below the expected value due to volatility drag ($\mu - \sigma^2/2$).
- The fan **widens** over time, showing increasing uncertainty.
- The upward skew is visible: the top of the fan is farther from the median than the bottom.

The gap between mean and median grows over time. After 2 years, the expected price is $S_0 e^{0.20} \approx 122.1$ while the median is $S_0 e^{0.16} \approx 117.4$ -- a gap of about $\$4.7$. For highly volatile assets ($\sigma = 0.5$), this gap becomes dramatic.

> **CFA Exam Tip:** Fan charts are used in economic forecasting (e.g., Bank of England inflation fan charts) and asset-liability management. Understanding that wider bands reflect higher uncertainty -- and that the bands grow as $\sqrt{T}$ -- is testable knowledge.> **Key Concept:** The fan chart illustrates a crucial and often counterintuitive result: the **expected value** of the stock price ($S_0 e^{\mu T}$) is NOT the most likely outcome. The most likely outcome (the mode of the lognormal) is $S_0 e^{(\mu - \sigma^2)T}$, which is BELOW the expected value. The mean is pulled up by rare but very large upside outcomes. This is the positive skew of the lognormal distribution.

> **Important:** This means that even under GBM with positive drift, the MEDIAN investor outcome is below the mean outcome. A few lucky paths pull the average up. This has implications for retirement planning and long-term wealth projections — using the expected (mean) value overstates what a typical investor will experience.


---
## 11. Calibration -- Estimating $\mu$ and $\sigma$ from Real Data

### The idea

Given prices $S_0, S_1, \ldots, S_n$, the log-returns are i.i.d. normal:
$$r_t = \ln(S_t/S_{t-1}) \sim \mathcal{N}\left((\mu - \sigma^2/2)\Delta t, \sigma^2 \Delta t\right)$$

### The estimation formulas

1. **Volatility:** $\hat{\sigma} = \text{Std}(r_t) / \sqrt{\Delta t}$ (annualize daily std)
2. **Drift:** $\hat{\mu} = \text{Mean}(r_t)/\Delta t + \hat{\sigma}^2/2$ (add back the Ito correction)

### Why the Ito correction matters for calibration

If you forget the $+\hat{\sigma}^2/2$ correction when estimating $\mu$, you get the *geometric* mean return, not the *arithmetic* mean return. For a stock with 20% volatility, this means underestimating the expected return by 2% per year -- a significant error for asset allocation.

### Worked example

252 daily log-returns with mean $\bar{r} = 0.0003$, std $s_r = 0.012$, $\Delta t = 1/252$:
- $\hat{\sigma} = 0.012 \times 15.87 = 0.190$ (19.0% annualized)
- $\hat{\mu} = 0.0003 \times 252 + 0.018 = 0.094$ (9.4% annualized)

> **Key Concept:** There is a deep asymmetry: volatility converges quickly (months of data), but drift requires decades. The standard error of $\hat{\sigma}$ decreases as $1/\sqrt{2n}$, while $\hat{\mu}$'s error decreases as $\sigma/\sqrt{n\Delta t}$ -- much slower. This is why option pricing depends on $\sigma$ (estimable), not $\mu$ (not reliably estimable).

### Practical implications

- **For option pricing:** Use implied volatility (market's estimate of $\sigma$). Drift does not matter because of risk-neutral pricing.
- **For portfolio management:** Use long-horizon data or factor models for $\mu$. Use recent data for $\sigma$.
- **For risk management (VaR):** Focus on $\sigma$ estimation, which is reliable with short data windows.

In [ ]:
def calibrate_gbm(prices, dt=1/252):
    """Calibrate GBM parameters from a price series.
    
    Step 1: Compute log-returns
    Step 2: Estimate sigma from the standard deviation of log-returns
    Step 3: Estimate mu from the mean of log-returns plus the sigma^2/2 correction
    """
    log_returns = np.diff(np.log(prices))
    sigma_hat = np.std(log_returns) / np.sqrt(dt)
    mu_hat = np.mean(log_returns) / dt + 0.5 * sigma_hat**2
    return mu_hat, sigma_hat

# Generate synthetic "historical" data with known parameters
true_mu, true_sigma = 0.12, 0.25
_, S_hist = gbm_exact(100, true_mu, true_sigma, 5.0, 5*252, 1)
prices = S_hist[0]

mu_cal, sigma_cal = calibrate_gbm(prices)

print(f"{'Parameter':<12} {'True':>10} {'Estimated':>10}")
print('-' * 34)
print(f"{'mu':<12} {true_mu:>10.4f} {mu_cal:>10.4f}")
print(f"{'sigma':<12} {true_sigma:>10.4f} {sigma_cal:>10.4f}")

**Interpretation:** Volatility estimate is close to truth, while drift deviates more. You need decades of data for drift, but only months for volatility.

This is not just a theoretical curiosity -- it has profound practical implications:
- **Portfolio managers** cannot reliably estimate expected returns from historical data alone. They must use economic theory, factor models, or forward-looking estimates.
- **Option traders** can reliably estimate volatility from even a few months of data. This is why the Black-Scholes model works in practice: it depends on $\sigma$ (reliable) and is independent of $\mu$ (unreliable).

Let us see how the estimates improve with more data.> **Key Concept:** This is one of the most important practical facts in quantitative finance: **volatility can be estimated precisely from short time series, but drift cannot.** The reason is mathematical:
> - $\hat{\sigma}$ converges at rate $1/\sqrt{n}$ where $n$ is the number of observations
> - $\hat{\mu}$ converges at rate $1/\sqrt{T}$ where $T$ is the total time span
>
> With daily data over 1 year, you have $n = 252$ observations (good for $\sigma$) but only $T = 1$ year of drift information (terrible for $\mu$). This is why option pricing uses $\sigma$ but doesn't need $\mu$ — and why the CAPM expected return is so hard to estimate empirically.


In [ ]:
# ── Convergence of estimators with increasing sample size
sample_sizes = [50, 100, 252, 504, 1260, 2520]
mu_ests = []
sig_ests = []
for n in sample_sizes:
    m_est, s_est = calibrate_gbm(prices[:n+1])
    mu_ests.append(m_est)
    sig_ests.append(s_est)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(sample_sizes, mu_ests, 'o-', color=PRIMARY, linewidth=2)
ax1.axhline(true_mu, color=SECONDARY, linestyle='--', label=f'True mu = {true_mu}')
ax1.set_xlabel('Sample Size (days)')
ax1.set_ylabel('Estimated mu')
ax1.set_title('Drift Estimation Convergence')
ax1.legend()

ax2.plot(sample_sizes, sig_ests, 'o-', color=PRIMARY, linewidth=2)
ax2.axhline(true_sigma, color=SECONDARY, linestyle='--', label=f'True sigma = {true_sigma}')
ax2.set_xlabel('Sample Size (days)')
ax2.set_ylabel('Estimated sigma')
ax2.set_title('Volatility Estimation Convergence')
ax2.legend()

plt.tight_layout()
plt.show()

**Interpretation:** Volatility (right) converges quickly even with 50 days. Drift (left) is much noisier and requires years to stabilize. This asymmetry has deep practical implications: option pricing depends on $\sigma$ (estimable), not $\mu$ (unreliable).

Merton (1980) showed that with $n$ observations spaced $\Delta t$ apart, the standard error of $\hat{\sigma}$ is $\sigma/\sqrt{2n}$ while the standard error of $\hat{\mu}$ is $\sigma/\sqrt{n \Delta t}$. For daily data over one year ($n = 252$, $\Delta t = 1/252$): SE($\hat{\sigma}$) $\approx 0.045\sigma$ but SE($\hat{\mu}$) $\approx \sigma$. The drift estimate has a standard error *equal to the volatility itself*!

> **CFA Exam Tip:** This calibration asymmetry is why the CFA curriculum emphasizes that historical returns are poor predictors of future returns, while historical volatility is a reasonable (though imperfect) predictor of future volatility. Risk estimation is fundamentally easier than return estimation.> **Important:** This asymmetry between drift and volatility estimation has profound implications:
> - **For option pricing (BSM):** Only $\sigma$ matters (not $\mu$), so options can be priced accurately
> - **For portfolio management (CAPM):** Expected returns ($\mu$) are needed but estimated poorly — this is why the "equity risk premium puzzle" and "expected return estimation" are among the hardest problems in finance
> - **For risk management (VaR):** Primarily needs $\sigma$, which is well-estimated — but tail behaviour requires more than GBM


---
## 12. Summary and Key Takeaways

| Concept | Key Result |
|---------|------------|
| Random walk | Symmetric, discrete; converges to Brownian motion via Donsker's theorem |
| Brownian motion | Continuous, $W_t \sim \mathcal{N}(0,t)$, quadratic variation = $t$ |
| Ito's lemma | Extra $\frac{1}{2}\sigma^2 f''$ correction vs ordinary calculus |
| GBM SDE | $dS/S = \mu \, dt + \sigma \, dW$ |
| GBM solution | $S_T = S_0 \exp[(\mu - \sigma^2/2)T + \sigma W_T]$ |
| Log-returns | Normal: $\ln(S_T/S_0) \sim \mathcal{N}((\mu - \sigma^2/2)T, \sigma^2 T)$ |
| Expected price | $E[S_T] = S_0 e^{\mu T}$ |
| Median price | $S_0 e^{(\mu - \sigma^2/2)T}$ (grows slower due to volatility drag) |
| Volatility drag | $\sigma^2/2$ gap between expected and median growth rates |
| Calibration | $\sigma$ is easy to estimate; $\mu$ requires decades of data |

### The big picture

GBM connects several deep ideas:
1. **Random walks** (discrete randomness) converge to **Brownian motion** (continuous randomness).
2. **Ito's lemma** provides the calculus rules for random processes, introducing a correction term absent in ordinary calculus.
3. **GBM** applies these tools to model stock prices as exponentials of Brownian motion, ensuring prices stay positive and returns are proportional.
4. The resulting **lognormal model** is the foundation of the Black-Scholes-Merton framework.

### Limitations of GBM

GBM is elegant but imperfect. Real stock returns exhibit:
- **Fat tails:** Extreme moves (crashes, rallies) occur more often than the normal distribution predicts.
- **Volatility clustering:** High-volatility days tend to follow high-volatility days.
- **Leverage effect:** Volatility tends to increase when prices fall.
- **Jumps:** Prices sometimes gap overnight due to news events.

These limitations motivate extensions like stochastic volatility (Heston model), jump-diffusion (Merton model), and GARCH models -- topics for future notebooks.### Limitations of GBM

GBM is the starting point, not the final word. Real stock prices exhibit:

| Empirical fact | GBM prediction | Reality |
|:---|:---|:---|
| Return distribution | Normal (log-returns) | Fat tails, negative skew |
| Volatility | Constant $\sigma$ | Time-varying (GARCH, stochastic vol) |
| Jumps | None (continuous paths) | Occasional large discontinuities |
| Correlation with vol | None | Leverage effect ($S \downarrow \Rightarrow \sigma \uparrow$) |

These limitations motivate extensions: stochastic volatility (Heston), jump-diffusion (Merton), and local volatility models — each building on the GBM foundation.


## 13. References

1. Bachelier, L. *Theorie de la Speculation*, 1900.
2. Samuelson, P. "Proof that Properly Anticipated Prices Fluctuate Randomly," *Industrial Management Review*, 1965.
3. Black, F. & Scholes, M. "The Pricing of Options and Corporate Liabilities," *JPE*, 1973.
4. Ito, K. "Stochastic Integral," *Proceedings of the Imperial Academy*, 1944.
5. Shreve, S. *Stochastic Calculus for Finance II*, Springer, 2004.
6. Hull, J.C. *Options, Futures, and Other Derivatives*, 11th ed., Pearson, 2022.

### Historical Significance

GBM is the foundation on which the entire edifice of mathematical finance is built. The Black-Scholes formula, risk-neutral pricing, and most derivatives models either use GBM directly or extend it. Understanding GBM deeply is therefore essential for anyone working in quantitative finance.
